In [1]:
import csv, json, os, time

import anthropic
import lancedb
import openai
from fastembed import TextEmbedding

DATA_DIR = "/workspace/data/filtered"
LANCEDB_PATH = "/workspace/lancedb_data"
TABLE_NAME = "raw_rows"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [2]:
def destroy_lancedb(path: str) -> None:
    """Drop every table so this notebook can be re-run from scratch."""
    db = lancedb.connect(path)
    names = db.table_names()
    for name in names:
        db.drop_table(name)
        print(f"Dropped table: {name}")
    print(f"Cleared {len(names)} table(s) from {path}" if names else f"No tables in {path}")


destroy_lancedb(LANCEDB_PATH)

Dropped table: raw_rows
Cleared 1 table(s) from /workspace/lancedb_data


/tmp/ipykernel_1430/2134705519.py:4: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  names = db.table_names()


In [3]:
# The four local sources. NOTHING has been resolved or geocoded - these are the
# rows exactly as the agencies published them.
SOURCES = [
    ("BURLINGAME",    "burlingame.tsv",                             "\t"),
    ("ABC-RETAIL",    "CA-ABC-LicenseReport-retail.csv",            ","),
    ("ABC-NONRETAIL", "CA-ABC-LicenseReport-nonretail.csv",         ","),
    ("CONTRACTORS",   "master-list-of-ca-licensed-contractors.csv", ","),
]

raw_rows = []
for source, filename, delim in SOURCES:
    with open(f"{DATA_DIR}/{filename}", newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f, delimiter=delim))
    raw_rows.extend((source, r) for r in rows)
    print(f"  {source:<14} {len(rows):>5} rows")

print(f"\n{len(raw_rows):,} raw rows total")

  BURLINGAME      4781 rows
  ABC-RETAIL       201 rows
  ABC-NONRETAIL     19 rows
  CONTRACTORS     3274 rows

8,275 raw rows total


In [4]:
def row_to_text(source: str, r: dict) -> str:
    """One raw row -> one human-readable string. No resolution, no coordinates."""
    if source == "BURLINGAME":
        return (f"City of Burlingame business licence. "
                f"Business name: {r['Business Name']}. "
                f"Address: {r['Address']}. "
                f"Licence {r['Account #']}, valid {r['Start Date']} to {r['Expire Date']}.")

    if source.startswith("ABC"):
        premises = r["Premises Addr."].split("Census Tract")[0].strip().rstrip(",")
        return (f"California ABC liquor licence. "
                f"Business name: {r['Business Name'] or r['Primary Owner']}. "
                f"Owner: {r['Primary Owner']}. Premises: {premises}. "
                f"Licence {r['License Number']}, type {r['License Type']}, "
                f"status {r['Status']}.")

    return (f"California contractor licence. "
            f"Business name: {r['BusinessName']}. "
            f"Address: {r['MailingAddress']}, {r['City']}, {r['State']} {r['ZIPCode']}. "
            f"Phone: {r['BusinessPhone']}. Licence {r['LicenseNo']}, "
            f"type {r['BusinessType']}, status {r['PrimaryStatus']}.")


print(row_to_text(*raw_rows[0]))
print()
print(row_to_text(*raw_rows[-1]))

City of Burlingame business licence. Business name: 101 MOBILITY. Address: 1731 ADRIAN RD STE 9, BURLINGAME, CA 94010-2109. Licence 20509315, valid 7/1/2025 to 6/30/2027.

California contractor licence. Business name: THE URBAN GARDENER. Address: 3639 - 21ST STREET, SAN FRANCISCO, CA 94114. Phone: (415) 681 9006. Licence 999606, type Corporation, status CLEAR.


In [5]:
# Baked into the image, so this is a local load, not a download.
print("Loading embedding model...", flush=True)
t0 = time.time()
embedding_model = TextEmbedding(EMBED_MODEL)
print(f"  ready in {time.time() - t0:.1f}s", flush=True)


def embed_one(text: str) -> list[float]:
    """fastembed returns a generator of ndarrays - take the first."""
    return next(iter(embedding_model.embed([text]))).tolist()

Loading embedding model...
  ready in 0.1s


In [6]:
texts = [row_to_text(s, r) for s, r in raw_rows]
print(f"Embedding {len(texts):,} rows...", flush=True)

t0 = time.time()
vectors = []
for i, v in enumerate(embedding_model.embed(texts, batch_size=64), 1):
    vectors.append(v)
    if i % 2000 == 0:
        print(f"  {i:,} / {len(texts):,}", flush=True)

elapsed = time.time() - t0
print(f"Done: {len(vectors):,} vectors in {elapsed:.0f}s ({len(vectors)/elapsed:.0f}/sec)", flush=True)


Embedding 8,275 rows...
  2,000 / 8,275
  4,000 / 8,275
  6,000 / 8,275
  8,000 / 8,275
Done: 8,275 vectors in 37s (227/sec)


In [7]:
records = [{"vector": v.tolist(), "text": t, "source": s}
           for v, t, (s, _) in zip(vectors, texts, raw_rows)]

db = lancedb.connect(LANCEDB_PATH)
table = db.create_table(TABLE_NAME, data=records)
print(f"LanceDB table '{TABLE_NAME}': {table.count_rows():,} rows")

LanceDB table 'raw_rows': 8,275 rows


In [8]:
# One of these needs a key in .env. After editing .env: docker compose up -d
anthropic_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") else None
openai_client    = openai.OpenAI()       if os.getenv("OPENAI_API_KEY")    else None

ANTHROPIC_MODEL = "claude-opus-5"
OPENAI_MODEL    = "gpt-5.4-nano"

print("Anthropic key present:", anthropic_client is not None)
print("OpenAI key present:   ", openai_client is not None)

Anthropic key present: True
OpenAI key present:    True


In [11]:
SYSTEM_PROMPT = (
    "Answer questions about California business licence records. You are given "
    "individual raw rows from four public datasets. These rows have NOT been "
    "entity-resolved: the same business may appear many times under slightly "
    "different names or addresses, and you cannot tell which rows describe the "
    "same real-world business. You have no coordinates, no distances, and no "
    "relationships between records. Answer only from the rows provided, and say "
    "so plainly when they are not enough."
)


def ask_raw(question: str, provider: str = "anthropic", top_k: int = 10) -> None:
    """RAG over raw, unresolved, ungeocoded rows."""
    print(f"\nQuestion: {question}\nProvider: {provider}")
    print("=" * 70)

    hits = table.search(embed_one(question)).limit(top_k).to_list()
    print(f"Retrieved {len(hits)} rows\n")

    context = "\n".join(f"- [{h['source']}] {h['text']}" for h in hits)
    user_message = f"Rows:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        r = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL, max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}])
        # Thinking models return reasoning blocks before the answer, so select
        # the text blocks rather than assuming content[0] is the reply.
        answer = "".join(b.text for b in r.content if b.type == "text")
    else:
        r = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": user_message}])
        answer = r.choices[0].message.content

    print(answer)

In [12]:
ask_raw("What businesses are near the Hyatt Regency San Francisco Airport?")


Question: What businesses are near the Hyatt Regency San Francisco Airport?
Provider: anthropic
Retrieved 10 rows

**Short answer: these rows can't tell me what's "near" anything.**

The records I have contain no coordinates, no distances, and no spatial relationships — just business names, owners, street addresses and licence details. Nothing in the data indicates which premises are close to which others.

What I *can* say is what the rows themselves show. The Hyatt Regency San Francisco Airport appears at **1333 Bayshore Hwy, Burlingame, CA 94010**, with:

- A City of Burlingame business licence (20500020, valid 5/24/1988 – 6/30/2027)
- Two active ABC liquor licences under licence number 220571 (types 47 and 58), owner HYATT CORPORATION

Other rows in this batch that happen to carry Burlingame or Millbrae addresses:

| Business | Address | Owner |
|---|---|---|
| Courtyard by Marriott San Francisco Airport – Burlingame | 765 Airport Blvd, Burlingame 94010 | Bayside 765 LLC |
| Hilto

In [13]:
ask_raw("How many separate licences does DeSilva Gates Construction hold, "
        "and is DESILVAGATES CONSTRUCTION the same company?")


Question: How many separate licences does DeSilva Gates Construction hold, and is DESILVAGATES CONSTRUCTION the same company?
Provider: anthropic
Retrieved 10 rows

## What the rows actually show

Ten distinct licence records share the address 11555 Dublin Blvd/Boulevard, Dublin, CA 94568 — nine state contractor licences and one City of Burlingame business licence:

**State contractor licences (CSLB)**

| Licence | Name on record | Type | Phone |
|---|---|---|---|
| 704195 | DE SILVA GATES CONSTRUCTION | Partnership | (925) 361 1533 |
| 1068502 | DE SILVA GATES CONSTRUCTION LP | Partnership | (925) 803 4277 |
| 1078219 | DESILVA GATES CONSTRUCTION LLC | Limited Liability | (925) 829 9220 |
| 1087418 | DESILVA GATES – PACIFIC STATES A JOINT VENTURE | JointVenture | (925) 829 9220 |
| 1093103 | DESILVA GATES CONSTRUCTION LLC – MCM A JOINT VENTURE | JointVenture | (925) 829 9220 |
| 1100766 | DE SILVA GATES CONSTRUCTION – MYERS A JOINT VENTURE | JointVenture | (925) 829 9220 |
| 1103207 

In [14]:
ask_raw("Which places within walking distance of 1333 Bayshore Hwy can serve a drink?")


Question: Which places within walking distance of 1333 Bayshore Hwy can serve a drink?
Provider: anthropic
Retrieved 10 rows

**I can't answer the "walking distance" part.** The rows I have contain no coordinates, no distances, and no way to compare one address to another geographically. I also have no row for 1333 Bayshore Hwy itself. Nothing here tells me how far any of these premises are from that address.

What I *can* do is list the rows that hold a licence type associated with serving alcohol, and let you judge the geography yourself:

**On Bayshore Hwy, Burlingame (same street name as your reference address):**
- H.L. PENINSULA HOT POT & GRILL LLC — 1600 Bayshore Hwy, Burlingame 94010-1504 — licence 634922, type 47, ACTIVE
- H.L. PENINSULA PEARL (owner: H.L. Peninsula Garden, Inc.) — 1590 Bayshore Hwy, Burlingame 94010-1601 — licence 589791, type 47, ACTIVE

**Elsewhere in Burlingame:**
- COURTYARD BY MARRIOTT SAN FRANCISCO AIRPORT – BURLINGAME (owner: Bayside 765 LLC) — 765 Ai